About the Dataset:

1. id: unique id for a news article
2. title: the title of a news article
3. author: author of the news article
4. text: the text of the article; could be incomplete
5. label: a label that marks whether the news article is real or fake:
           1: Fake news
           0: real News

Code  ----->>>	কাজ
re	  ---->>>text clean করা
stopwords  ----->	useless words remove করা
PorterStemmer	-----> word root বানানো

In [13]:
import numpy as np
import pandas as pd
import nltk 
import re    #Text এর মধ্যে pattern খুঁজে বের করা বা replace করা। 
from nltk.corpus import stopwords    #Common words যেগুলোর meaning কম থাকে NLP-তে:  
from nltk.stem.porter import PorterStemmer   #Word কে তার root form এ আনা।
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ataur\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [14]:
df = pd.read_csv('E:/ML/fake_news_dataset.csv')

In [15]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [16]:
df.isnull().sum()

id           0
title      558
author    1957
text        39
label        0
dtype: int64

In [18]:
# replacing the null values with empty string
df= df.fillna('')

In [19]:
df['content'] = df['author'] + df['title']

In [20]:
print(df)

          id                                              title  \
0          0  House Dem Aide: We Didn’t Even See Comey’s Let...   
1          1  FLYNN: Hillary Clinton, Big Woman on Campus - ...   
2          2                  Why the Truth Might Get You Fired   
3          3  15 Civilians Killed In Single US Airstrike Hav...   
4          4  Iranian woman jailed for fictional unpublished...   
...      ...                                                ...   
20795  20795  Rapper T.I.: Trump a ’Poster Child For White S...   
20796  20796  N.F.L. Playoffs: Schedule, Matchups and Odds -...   
20797  20797  Macy’s Is Said to Receive Takeover Approach by...   
20798  20798  NATO, Russia To Hold Parallel Exercises In Bal...   
20799  20799                          What Keeps the F-35 Alive   

                                          author  \
0                                  Darrell Lucus   
1                                Daniel J. Flynn   
2                             Consortiu

In [22]:
X = df.drop(columns='label',axis =1)
Y = df['label']

In [23]:
print(X)

          id                                              title  \
0          0  House Dem Aide: We Didn’t Even See Comey’s Let...   
1          1  FLYNN: Hillary Clinton, Big Woman on Campus - ...   
2          2                  Why the Truth Might Get You Fired   
3          3  15 Civilians Killed In Single US Airstrike Hav...   
4          4  Iranian woman jailed for fictional unpublished...   
...      ...                                                ...   
20795  20795  Rapper T.I.: Trump a ’Poster Child For White S...   
20796  20796  N.F.L. Playoffs: Schedule, Matchups and Odds -...   
20797  20797  Macy’s Is Said to Receive Takeover Approach by...   
20798  20798  NATO, Russia To Hold Parallel Exercises In Bal...   
20799  20799                          What Keeps the F-35 Alive   

                                          author  \
0                                  Darrell Lucus   
1                                Daniel J. Flynn   
2                             Consortiu

Stemming: Stemming is the process of reducing a word to its Root word
example: actor, actress, acting --> act

In [28]:
port_stem = PorterStemmer()

In [29]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)  #content থেকে সব কিছু বাদ দিচ্ছে যেগুলো letter (a–z, A–Z) না। যেগুলো remove হবে, সেগুলোর জায়গায় space বসাবে
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()  #এই লাইনের কাজ হলো একটা string কে শব্দে (word/token) ভাগ করা।
    stemmed_content = [port_stem.stem(word) for word in stemmed_content
                        if not word  in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)  # লাইনটার কাজ হলো list-এর সব শব্দকে আবার একসাথে string বানানো। ' ' = space (শব্দগুলোর মাঝে space দিবে)
    return stemmed_content


stemmed_content_new = []
for word in stemmed_content:
    if word not in stopwords.words('english'):
        stemmed_content_new.append(port_stem.stem(word))
stemmed_content = stemmed_content_new

''''''''''''''''''''''''''''''''''''''''''''

stemmed_content = [port_stem.stem(word) for word in stemmed_content 
                       if not word in stopwords.words('english')]
two type of code do same work

In [30]:
df['content'] = df['content'].apply(stemming)

In [31]:
print(df['content'])

0        darrel lucushous dem aid even see comey letter...
1        daniel j flynnflynn hillari clinton big woman ...
2                consortiumnew comwhi truth might get fire
3        jessica purkiss civilian kill singl us airstri...
4        howard portnoyiranian woman jail fiction unpub...
                               ...                        
20795    jerom hudsonrapp trump poster child white supr...
20796    benjamin hoffmann f l playoff schedul matchup ...
20797    michael j de la merc rachel abramsmaci said re...
20798    alex ansarynato russia hold parallel exercis b...
20799                        david swansonwhat keep f aliv
Name: content, Length: 20800, dtype: object


In [ ]:
# df['content']   ---->	Pandas Series
# df['content'].values  ---> NumPy array
X = df['content'].values
Y = df['label'].values

In [40]:
print(X)

['darrel lucushous dem aid even see comey letter jason chaffetz tweet'
 'daniel j flynnflynn hillari clinton big woman campu breitbart'
 'consortiumnew comwhi truth might get fire' ...
 'michael j de la merc rachel abramsmaci said receiv takeov approach hudson bay new york time'
 'alex ansarynato russia hold parallel exercis balkan'
 'david swansonwhat keep f aliv']


In [41]:
print(Y)

[1 0 1 ... 0 1 1]


Tf - Idf
TF-IDF মানে কী?
TF (Term Frequency) --> একটা word কতবার এসেছে
IDF (Inverse Document Frequency) --> wordটা কত rare / important

In [ ]:
# convert the textual data to Feature Vectors
vectorizer = TfidfVectorizer()  

In [44]:
vectorizer.fit(X)
X = vectorizer.transform(X)

vectorizer.fit(X)
👉 এখানে model শিখছে:
কোন শব্দ কতবার এসেছে (TF)
কোন শব্দ কতটা important (IDF)
📌 এটা শুধু vocabulary তৈরি করে:
সব unique words
তাদের importance


X = vectorizer.transform(X)
👉 এখন আসল কাজ:
text → numerical matrix
প্রতিটা document → vector

In [45]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 198373 stored elements and shape (20800, 28419)>
  Coords	Values
  (0, 578)	0.2694167078545384
  (0, 4211)	0.36253203231506576
  (0, 5006)	0.24725958235728157
  (0, 5969)	0.35488202138141456
  (0, 6273)	0.2839932825877812
  (0, 8022)	0.2313366174248873
  (0, 12782)	0.24619727512767192
  (0, 14555)	0.2917725968420029
  (0, 15019)	0.4300622675963931
  (0, 22724)	0.25523360180691607
  (0, 26340)	0.2808837940159642
  (1, 2622)	0.3562953366945267
  (1, 3281)	0.18652439327549428
  (1, 3859)	0.45980466668763476
  (1, 4767)	0.23338756776626793
  (1, 5916)	0.31810058109638056
  (1, 8772)	0.5258635625386451
  (1, 11313)	0.24166773097712638
  (1, 27923)	0.36911845953845024
  (2, 5121)	0.5511414848555652
  (2, 5240)	0.40440534260277944
  (2, 8567)	0.3411947414020896
  (2, 9454)	0.30743020569262086
  (2, 16361)	0.43295215406038445
  (2, 26235)	0.3665032495181434
  :	:
  (20797, 1249)	0.3072223353708335
  (20797, 2257)	0.3357782642976524
